# 02 — Leakage-safe dataset generation

Assign font families to train, validation, and test before generating images. Then render varied text crops and verify that no family overlaps.

# Setup

Run this notebook from the repository root. In Google Colab, clone the GitHub repository first and replace the placeholder URL.

In [ ]:
from pathlib import Path
import os, sys

REPO_URL = "PASTE_YOUR_GITHUB_REPOSITORY_URL_HERE"
if 'google.colab' in sys.modules:
    if not Path('/content/fontsense-capstone').exists():
        if 'PASTE_' in REPO_URL:
            raise ValueError('Replace REPO_URL with your GitHub repository URL first.')
        !git clone {REPO_URL} /content/fontsense-capstone
    os.chdir('/content/fontsense-capstone')
    %pip install -q -r requirements.txt
    %pip install -q -e .
else:
    root = Path.cwd()
    if root.name == 'notebooks':
        root = root.parent
    os.chdir(root)
    os.environ['PYTHONPATH'] = str(root / 'src') + os.pathsep + os.environ.get('PYTHONPATH', '')
    if str(root / 'src') not in sys.path:
        sys.path.insert(0, str(root / 'src'))
print('Project root:', Path.cwd())


In [ ]:
FONT_MANIFEST = 'data/interim/google_fonts_manifest.csv'  # use system_fonts_manifest.csv for a small proof
OUTPUT_DIR = 'data/processed/fontsense_google'

In [ ]:
!python -m fontsense.generate_dataset --font-manifest {FONT_MANIFEST} --output-dir {OUTPUT_DIR} --images-per-family 40

In [ ]:
import pandas as pd
families = pd.read_csv(f'{OUTPUT_DIR}/families.csv')
manifest = pd.read_csv(f'{OUTPUT_DIR}/manifest.csv')
display(families.groupby(['category','split']).size().unstack(fill_value=0))
assert families.groupby('family')['split'].nunique().max() == 1
print('No family leakage. Images:', len(manifest))

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
sample = manifest.groupby('category', group_keys=False).sample(2, random_state=42)
fig, axes = plt.subplots(5, 2, figsize=(10, 12))
for ax, (_, row) in zip(axes.ravel(), sample.iterrows()):
    ax.imshow(Image.open(row.image_path))
    ax.set_title(f"{row.category} | {row.family}")
    ax.axis('off')
plt.tight_layout()